In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import chi2_contingency, f_oneway
from statsmodels.stats.weightstats import ztest

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import ConfusionMatrixDisplay

import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("data/Instagram_Analytics.csv")

print("Social Media Engagement Analysis")
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("Dataset information")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().copy()
df["post_datetime"] = pd.to_datetime(df["post_datetime"], errors="coerce")

print("\nPerformance category counts:")
display(df["performance_bucket_label"].value_counts())

print("\nNumerical summary:")
display(df.describe().round(3))

print("\nUnique accounts:", df["account_id"].nunique())

In [ ]:
numerical_columns = [
    "likes", "comments", "shares", "saves",
    "reach", "impressions", "engagement_rate",
    "followers_gained", "caption_length", "hashtags_count"
]

print("Mean:")
display(df[numerical_columns].mean().round(3))

print("Median:")
display(df[numerical_columns].median().round(3))

print("Standard deviation:")
display(df[numerical_columns].std().round(3))

print("Mode:")
display(df[numerical_columns].mode().iloc[0])

plt.figure(figsize=(8, 5))
sns.histplot(df["engagement_rate"], bins=30, kde=True)
plt.title("Distribution of Engagement Rate")
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="media_type")
plt.title("Posts by Media Type")
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="media_type", y="engagement_rate")
plt.title("Engagement Rate by Media Type")
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="reach", y="likes", alpha=0.5)
plt.title("Reach vs Likes")
plt.show()

In [ ]:
sample_size = min(100, len(df))

# Simple random sampling without replacement
random_sample = df.sample(n=sample_size, random_state=42)

# Simple random sampling with replacement
replacement_sample = df.sample(
    n=sample_size, replace=True, random_state=42
)

# Sampling fraction
sampling_fraction = sample_size / len(df)

# Systematic sampling
step = max(1, len(df) // sample_size)
systematic_sample = df.iloc[::step].head(sample_size)

# Stratified sampling by account type
stratified_sample = (
    df.groupby("account_type", group_keys=False)
    .sample(frac=0.20, random_state=42)
)

# Cluster sampling by account
rng = np.random.default_rng(42)
accounts = df["account_id"].unique()
selected_accounts = rng.choice(
    accounts, size=max(1, int(np.ceil(len(accounts) * 0.2))),
    replace=False
)
cluster_sample = df[df["account_id"].isin(selected_accounts)]

print("Population size:", len(df))
print("Random sample:", len(random_sample))
print("Sample with replacement:", len(replacement_sample))
print("Sampling fraction:", round(sampling_fraction, 4))
print("Systematic sample:", len(systematic_sample))
print("Stratified sample:", len(stratified_sample))
print("Cluster sample:", len(cluster_sample))

print("\nPopulation mean:", df["engagement_rate"].mean())
print("Random sample mean:", random_sample["engagement_rate"].mean())
print(
    "Absolute sampling error:",
    abs(df["engagement_rate"].mean() - random_sample["engagement_rate"].mean())
)

In [ ]:
engagement = df["engagement_rate"].dropna()

print("Mean:", engagement.mean())
print("Standard deviation:", engagement.std())

# Normal distribution
plt.figure(figsize=(8, 5))
sns.histplot(engagement, bins=30, kde=True)
plt.title("Engagement Rate Distribution")
plt.show()

# Binomial distribution: whether a post has a call to action
p = df["has_call_to_action"].mean()
n = 10

x = np.arange(0, n + 1)
probabilities = stats.binom.pmf(x, n, p)

plt.figure(figsize=(8, 5))
plt.bar(x, probabilities)
plt.title("Binomial Distribution")
plt.xlabel("Number of posts with a call to action in 10 trials")
plt.ylabel("Probability")
plt.show()

# Poisson distribution: illustrative model for comment counts
comment_mean = df["comments"].mean()
x = np.arange(0, 31)

plt.figure(figsize=(8, 5))
plt.bar(x, stats.poisson.pmf(x, comment_mean))
plt.title("Poisson Distribution for Comment Counts")
plt.xlabel("Number of comments")
plt.ylabel("Probability")
plt.show()

# Uniform distribution: theoretical example
x_uniform = np.linspace(0, 1, 100)

plt.figure(figsize=(8, 5))
plt.plot(x_uniform, stats.uniform.pdf(x_uniform))
plt.title("Uniform Distribution from 0 to 1")
plt.xlabel("Value")
plt.ylabel("Density")
plt.show()

# Central Limit Theorem
rng = np.random.default_rng(42)
sample_means = [
    rng.choice(engagement, size=30, replace=True).mean()
    for _ in range(1000)
]

print("Original population mean:", engagement.mean())
print("Mean of sample means:", np.mean(sample_means))
print("Standard error of sample means:", np.std(sample_means, ddof=1))

plt.figure(figsize=(8, 5))
sns.histplot(sample_means, bins=30, kde=True)
plt.title("Central Limit Theorem: Distribution of Sample Means")
plt.xlabel("Sample mean engagement rate")
plt.show()

In [ ]:
data = df["engagement_rate"].dropna()

n = len(data)
sample_mean = data.mean()
standard_error = stats.sem(data)

ci = stats.t.interval(
    confidence=0.95,
    df=n - 1,
    loc=sample_mean,
    scale=standard_error
)

print("Point estimate:", sample_mean)
print("Standard error:", standard_error)
print("95% confidence interval:", ci)

# Reference value used for demonstration, not an official benchmark
reference_mean = 0.04
alpha = 0.05

# One-sample Z-test
z_stat, z_p = ztest(data, value=reference_mean)

print("\nZ-test statistic:", z_stat)
print("Z-test p-value:", z_p)

if z_p < alpha:
    print("Z-test: Reject the null hypothesis.")
else:
    print("Z-test: Fail to reject the null hypothesis.")

# One-sample t-test
t_stat, t_p = stats.ttest_1samp(data, popmean=reference_mean)

print("\nt-test statistic:", t_stat)
print("t-test p-value:", t_p)

if t_p < alpha:
    print("t-test: Reject the null hypothesis.")
else:
    print("t-test: Fail to reject the null hypothesis.")

In [ ]:
# Pearson and Spearman correlation
pairs = [
    ("reach", "likes"),
    ("likes", "comments"),
    ("shares", "saves"),
    ("hashtags_count", "engagement_rate")
]

for col1, col2 in pairs:
    pair_data = df[[col1, col2]].dropna()

    pearson_r, pearson_p = stats.pearsonr(
        pair_data[col1], pair_data[col2]
    )
    spearman_r, spearman_p = stats.spearmanr(
        pair_data[col1], pair_data[col2]
    )

    print(f"\n{col1} vs {col2}")
    print("Pearson correlation:", pearson_r, "p-value:", pearson_p)
    print("Spearman correlation:", spearman_r, "p-value:", spearman_p)

# Correlation heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(
    df.select_dtypes(include="number").corr(),
    annot=True, cmap="coolwarm", fmt=".2f"
)
plt.title("Correlation Heatmap")
plt.show()

# Chi-square test of independence
table = pd.crosstab(
    df["account_type"], df["performance_bucket_label"]
)

chi2, chi_p, dof, expected = chi2_contingency(table)

print("\nChi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("Chi-square p-value:", chi_p)

# One-way ANOVA
groups = [
    group["engagement_rate"].dropna().values
    for _, group in df.groupby("media_type")
]

f_stat, anova_p = f_oneway(*groups)

print("\nANOVA F-statistic:", f_stat)
print("ANOVA p-value:", anova_p)

In [ ]:
categorical_features = [
    "account_type",
    "media_type",
    "content_category",
    "traffic_source",
    "day_of_week"
]

numeric_features = [
    "follower_count",
    "has_call_to_action",
    "post_hour",
    "caption_length",
    "hashtags_count"
]

features = categorical_features + numeric_features
target = "performance_bucket_label"

model_data = df[features + [target]].dropna()

X = model_data[features]
y = model_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    ),
    ("numeric", StandardScaler(), numeric_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print("\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, predictions, xticks_rotation=45
)
plt.title("Performance Classification")
plt.show()

In [ ]:
print("=" * 50)
print("PROJECT SUMMARY")
print("=" * 50)

print("Total posts analysed:", len(df))
print("Unique accounts:", df["account_id"].nunique())
print("Average engagement rate:", round(df["engagement_rate"].mean(), 4))
print("Most common media type:", df["media_type"].mode()[0])
print("Most common performance category:",
      df["performance_bucket_label"].mode()[0])
print("ML test accuracy:", round(accuracy_score(y_test, predictions), 4))

print("\nAll statistical practicals completed.")